- 2-3: Skip List by Shusen Wang: https://youtu.be/UGaOXaXAM5M?si=V0tRonejf6ei-W3S
- MIT OpenCourseWare, Design and Analysis of Algorithms:Lecture 7: Randomization: Skip Lists: https://ocw.mit.edu/courses/6-046j-design-and-analysis-of-algorithms-spring-2015/resources/lecture-7-randomization-skip-lists/

In [6]:

from random import randrange

class Stack:
    """
    A simple stack implementation used during insertion.
    
    Why a stack?
    When searching for an insertion point in a skip list, we move down levels 
    and across nodes. We need to save the exact header/data nodes where we 
    dropped down to a lower level, so we can process them in reverse order 
    when building towers. A stack's Last-In, First-Out (LIFO) property is perfect for this.
    """
    def __init__(self):
        self._items = []

    def is_empty(self):
        return len(self._items) == 0

    def push(self, item):
        self._items.append(item)

    def pop(self):
        return self._items.pop()

    def peek(self):
        return self._items[-1]

    def size(self):
        return len(self._items)


class HeaderNode:
    """
    Represents a header node in the vertical leftmost column of the skip list.
    
    Revision Note:
    - Header nodes form the vertical 'spine' on the far left.
    - `next`: Points to the first data node in that specific horizontal level.
    - `down`: Points to the corresponding header node on the next lower level.
    """
    def __init__(self):
        self._next = None
        self._down = None

    @property
    def next(self):
        return self._next

    @next.setter
    def next(self, value):
        self._next = value

    @property
    def down(self):
        return self._down

    @down.setter
    def down(self, value):
        self._down = value


class DataNode:
    """
    Represents an individual data node storing a key-value pair.
    
    Revision Note:
    - Each node holds its own key, data value, and two pointers:
      - `next`: Points to the next node horizontally on the same level.
      - `down`: Points to the corresponding node in the tower directly below it.
    """
    def __init__(self, key, value):
        self._key = key
        self._data = value
        self._next = None
        self._down = None

    @property
    def key(self):
        return self._key

    @property
    def data(self):
        return self._data

    @data.setter
    def data(self, value):
        self._data = value

    @property
    def next(self):
        return self._next

    @next.setter
    def next(self, value):
        self._next = value

    @property
    def down(self):
        return self._down

    @down.setter
    def down(self, value):
        self._down = value


class SkipList:
    """
    Implements the core Skip List data structure with probabilistic leveling,
    efficient O(log n) search, and insertion operations.
    """
    def __init__(self):
        # The entry point of the skip list (top-left header node)
        self._head = None

    def search(self, key):
        """
        Searches for a key in the skip list and returns its associated value.
        
        Algorithm logic:
        1. Start at the top-left head of the skip list.
        2. Look right (`current.next`):
           - If there is no next node on this level, drop down (`current = current.down`).
           - If the next node's key matches our target key, return its data value!
           - If our target key is LESS than the next node's key, we've gone too far; drop down.
           - If our target key is GREATER than the next node's key, move right (`current = current.next`).
        3. Repeat until found or we hit the bottom (returning None).
        """
        current = self._head
        while current:
            if current.next is None:
                current = current.down
            else:
                if current.next.key == key:
                    return current.next.data  # Found match!
                if key < current.next.key:
                    current = current.down  # Target is smaller, drop down level
                else:
                    current = current.next  # Target is larger, move right
        return None  # Key not found in skip list

    def insert(self, key, value):
        """
        Inserts a new key-value pair into the skip list.
        
        Two main steps:
        1. Find the correct insertion points across levels using a stack.
        2. Insert the new node at level 0, then use randomized "coin flips" 
           (randrange) to probabilistically build its tower upward.
        """
        if self._head is None:
            # Case 1: Empty skip list initialization
            self._head = HeaderNode()
            temp = DataNode(key, value)
            self._head.next = temp
            top = temp
            # Probabilistically grow the tower upwards via coin flips (1 = heads)
            while randrange(2) == 1:
                newhead = HeaderNode()
                temp = DataNode(key, value)
                temp.down = top
                newhead.next = temp
                newhead.down = self._head
                self._head = newhead
                top = temp
        else:
            # Case 2: Non-empty skip list search & insert
            tower = Stack()
            current = self._head
            
            # Traverse to find where the key fits, pushing drop-down points onto the stack
            while current:
                if current.next is None:
                    tower.push(current)
                    current = current.down
                else:
                    if current.next.key > key:
                        tower.push(current)
                        current = current.down
                    else:
                        current = current.next

            # Insert at level 0 (pop the bottom-most insertion point from stack)
            lowest_level = tower.pop()
            temp = DataNode(key, value)
            temp.next = lowest_level.next
            lowest_level.next = temp
            top = temp
            
            # Probabilistically build towers upward layer by layer
            while randrange(2) == 1:
                if tower.is_empty():
                    # If we exceeded current height, create a new top header row
                    newhead = HeaderNode()
                    temp = DataNode(key, value)
                    temp.down = top
                    newhead.next = temp
                    newhead.down = self._head
                    self._head = newhead
                    top = temp
                else:
                    # Pop the next level up from our stack and insert the tower node
                    next_level = tower.pop()
                    temp = DataNode(key, value)
                    temp.down = top
                    temp.next = next_level.next
                    next_level.next = temp
                    top = temp

    def display(self):
        """
        Traversal method to print out the skip list layer by layer (top down).
        Useful for visually debugging and verifying towers and levels.
        """
        current_head = self._head
        
        # Calculate total height to label levels accurately
        height = 0
        temp_h = self._head
        while temp_h:
            height += 1
            temp_h = temp_h.down
            
        current_level_num = height - 1
        current = current_head
        
        while current:
            print(f"Level {current_level_num}: ", end="")
            node = current.next
            while node:
                print(f"({node.key}: {node.data})", end=" -> ")
                node = node.next
            print("None")
            
            current = current.down
            current_level_num -= 1


class Map:
    """
    Implements the Map Abstract Data Type (ADT) wrapping our internal SkipList collection.
    Adheres strictly to the required Map operations: __init__, put(key, value), and get(key).
    """
    def __init__(self):
        self.collection = SkipList()

    def put(self, key, value):
        self.collection.insert(key, value)

    def get(self, key):
        return self.collection.search(key)

    def display(self):
        self.collection.display()

In [4]:
from random import randrange

class Stack:
    """
    A simple stack implementation used during insertion.
    
    Why a stack?
    When searching for an insertion point in a skip list, we move down levels 
    and across nodes. We need to save the exact header or data nodes where we 
    dropped down to a lower level, so we can process them in reverse order 
    when building towers. A stack's Last-In, First-Out (LIFO) property is perfect for this.
    """
    def __init__(self):
        self._items = []

    def is_empty(self):
        return len(self._items) == 0

    def push(self, item):
        self._items.append(item)

    def pop(self):
        return self._items.pop()

    def peek(self):
        return self._items[-1]

    def size(self):
        return len(self._items)


class HeaderNode:
    """
    Represents a header node in the vertical leftmost column of the skip list.
    
    Revision Note:
    - Header nodes form the vertical 'spine' on the far left.
    - `next`: Points to the first data node in that specific horizontal level.
    - `down`: Points to the corresponding header node on the next lower level.
    """
    def __init__(self):
        self._next = None
        self._down = None

    @property
    def next(self):
        return self._next

    @next.setter
    def next(self, value):
        self._next = value

    @property
    def down(self):
        return self._down

    @down.setter
    def down(self, value):
        self._down = value


class DataNode:
    """
    Represents an individual data node storing a key-value pair.
    
    Revision Note:
    - Each node holds its own key, data value, and two pointers:
      - `next`: Points to the next node horizontally on the same level.
      - `down`: Points to the corresponding node in the tower directly below it.
    """
    def __init__(self, key, value):
        self._key = key
        self._data = value
        self._next = None
        self._down = None

    @property
    def key(self):
        return self._key

    @property
    def data(self):
        return self._data

    @data.setter
    def data(self, value):
        self._data = value

    @property
    def next(self):
        return self._next

    @next.setter
    def next(self, value):
        self._next = value

    @property
    def down(self):
        return self._down

    @down.setter
    def down(self, value):
        self._down = value


class SkipList:
    """
    Implements the core Skip List data structure with probabilistic leveling,
    efficient O(log n) search, insertion operations, and a traversal display method.
    """
    def __init__(self):
        # The entry point of the skip list (top-left header node)
        self._head = None

    def search(self, key):
        """
        Searches for a key in the skip list and returns its associated value.
        
        Algorithm logic:
        1. Start at the top-left head of the skip list.
        2. Look right (`current.next`):
           - If there is no next node on this level, drop down (`current = current.down`).
           - If the next node's key matches our target key, return its data value!
           - If our target key is LESS than the next node's key, we've gone too far; drop down.
           - If our target key is GREATER than the next node's key, move right (`current = current.next`).
        3. Repeat until found or we hit the bottom (returning None).
        """
        current = self._head
        while current:
            if current.next is None:
                current = current.down
            else:
                if current.next.key == key:
                    return current.next.data  # Found match!
                if key < current.next.key:
                    current = current.down  # Target is smaller, drop down level
                else:
                    current = current.next  # Target is larger, move right
        return None  # Key not found in skip list

    def insert(self, key, value):
        """
        Inserts a new key-value pair into the skip list.
        
        Two main steps:
        1. Find the correct insertion points across levels using a stack.
        2. Insert the new node at level 0, then use randomized "coin flips" 
           (randrange) to probabilistically build its tower upward.
        """
        if self._head is None:
            # Case 1: Empty skip list initialization
            self._head = HeaderNode()
            temp = DataNode(key, value)
            self._head.next = temp
            top = temp
            # Probabilistically grow the tower upwards via coin flips (1 = heads)
            while randrange(2) == 1:
                newhead = HeaderNode()
                temp = DataNode(key, value)
                temp.down = top
                newhead.next = temp
                newhead.down = self._head
                self._head = newhead
                top = temp
        else:
            # Case 2: Non-empty skip list search & insert
            tower = Stack()
            current = self._head
            
            # Traverse to find where the key fits, pushing drop-down points onto the stack
            while current:
                if current.next is None:
                    tower.push(current)
                    current = current.down
                else:
                    if current.next.key > key: #this means that our key fall in between??
                        tower.push(current)
                        current = current.down
                    else:
                        current = current.next

            # Insert at level 0 (pop the bottom-most insertion point from stack)
            lowest_level = tower.pop()
            temp = DataNode(key, value)
            temp.next = lowest_level.next
            lowest_level.next = temp
            top = temp
            
            # Probabilistically build towers upward layer by layer
            while randrange(2) == 1:
                if tower.is_empty():
                    # If we exceeded current height, create a new top header row
                    newhead = HeaderNode()
                    temp = DataNode(key, value)
                    temp.down = top
                    newhead.next = temp
                    newhead.down = self._head
                    self._head = newhead
                    top = temp
                else:
                    # Pop the next level up from our stack and insert the tower node
                    next_level = tower.pop()
                    temp = DataNode(key, value)
                    temp.down = top
                    temp.next = next_level.next
                    next_level.next = temp
                    top = temp

    def display(self):
        """
        Traversal method to print out the skip list layer by layer (top down).
        Useful for visually debugging and verifying towers and levels.
        """
        current_head = self._head
        
        # Calculate total height to label levels accurately
        height = 0
        temp_h = self._head
        while temp_h:
            height += 1
            temp_h = temp_h.down
            
        current_level_num = height - 1
        current = current_head
        
        while current:
            print(f"Level {current_level_num}: ", end="")
            node = current.next
            while node:
                print(f"({node.key}: {node.data})", end=" -> ")
                node = node.next
            print("None")
            
            current = current.down
            current_level_num -= 1


class Map:
    """
    Implements the Map Abstract Data Type (ADT) wrapping our internal SkipList collection.
    Adheres strictly to the required Map operations: __init__, put(key, value), and get(key).
    """
    def __init__(self):
        self.collection = SkipList()

    def put(self, key, value):
        self.collection.insert(key, value)

    def get(self, key):
        return self.collection.search(key)

    def display(self):
        self.collection.display()




In [8]:
# Create a new Map instance
my_map = Map()

# Insert key-value pairs
my_map.put(17, "to")
my_map.put(31, "it")
my_map.put(54, "on")
my_map.put(77, "of")
my_map.put(93, "be")

# Retrieve values
print("Get 77:", my_map.get(77))  # Output: of
print("Get 31:", my_map.get(31))  # Output: it

print("\n--- Visualizing Skip List Levels ---")
my_map.display()

Get 77: of
Get 31: it

--- Visualizing Skip List Levels ---
Level 4: (93: be) -> None
Level 3: (93: be) -> None
Level 2: (93: be) -> None
Level 1: (54: on) -> (93: be) -> None
Level 0: (17: to) -> (31: it) -> (54: on) -> (77: of) -> (93: be) -> None
